# Western Balkan Studies

## Define RUN TAG

In [ ]:
RUN_ID = 'vre_low_20260310'

* load packages

In [ ]:
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

import RES.visuals as vis
from RES import utility as utils
from RES.hdf5_handler import DataHandler

plt.style.use('../RES/visual_styles/elsevier.mplstyle')
# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning)


- Load Configs

In [ ]:
cfg=utils.load_config('../config/config_WB6.yaml')
run_id=cfg.get('Scenario').get('run_id')


sub_national_unit_tag=cfg.get('GADM').get('datafield_mapping').get('NAME_2')
country_name=cfg.get('country','Western Balkan Region') # type: ignore
country_kwd=country_name.replace(' ','')
CRS_m = cfg.get('default_CRS').get('meters')  # Default metric CRS
CRS_d = cfg.get('default_CRS').get('degrees')  # Default geographic CRS
regions=['AL','BA','XK','ME','RS','MK']  #'AL','BA','XK','ME','MK','RS'

vis_save_to_root=utils.ensure_path(f"../vis/{country_kwd}/{RUN_ID}/WB6_fullRegion")

## Load Store

In [ ]:
 #All the regions should have RUN_ID results available
WB6_store = {}
utils.print_update(level=1,message=f"Loading data stores for WB6 regions with RUN_ID: {RUN_ID} and Regions: {regions}")
for region in regions:
    country_dict = {}
    try:
        store = Path(f"../data/store/{country_kwd}/resources_{country_kwd}_{region}_{RUN_ID}.h5")
        assert store.exists(), f"Store path doesn't exist: {store}"
        res_data = DataHandler(store, show_structure=False)
        country_dict['cells'] = res_data.from_store('cells')
        country_dict['boundary'] = res_data.from_store('boundary')
        country_dict['lines'] = res_data.from_store('lines')
        # country_dict['LandAvailability'] = res_data.from_store('LandAvailability')
        WB6_store[region] = country_dict
        utils.print_update(level=2,message=f"✓ Loaded data for region: {region}") 
    except Exception as e:
        print(f"X Error with region {region}: {e}")
        continue

## Load All Cells

In [ ]:
all_cells_df=[WB6_store[region]['cells'] for region in WB6_store]
WB6_cells = gpd.GeoDataFrame(pd.concat(all_cells_df, ignore_index=False), crs=all_cells_df[0].crs)

# Load Test/Validation data

In [ ]:
existing_VREs_data_path=Path("../data/validation_data/existing_VREs_WB6.csv")
if existing_VREs_data_path.exists():
    existing_VREs=pd.read_csv(existing_VREs_data_path)
    existing_VREs_gdf=gpd.GeoDataFrame(existing_VREs,geometry=gpd.points_from_xy(existing_VREs.Longitude,existing_VREs.Latitude),crs="EPSG:4326")
    utils.print_update(level=1,message=f"✓ Loaded validation data for existing VREs from {existing_VREs_data_path}")
else:
    existing_VREs_gdf=None
    utils.print_warning(f"Validation data for existing VREs not found at {existing_VREs_data_path}")

# Boundary

- Prepare WB6 boundary

In [ ]:
# Combine all region boundaries into a single GeoDataFrame
boundary_gdfs = [WB6_store[region]['boundary'] for region in WB6_store]
WB6_boundary = gpd.GeoDataFrame(pd.concat(boundary_gdfs, ignore_index=True), crs=boundary_gdfs[0].crs)
WB6_boundary_dissolved = WB6_boundary.dissolve(by="Country")[["geometry"]].reset_index()

- Process the boundary info for raster plotting

In [ ]:
WB6_boundary_dissolved_reproj=WB6_boundary_dissolved.to_crs(CRS_m)
# Get total bounds from boundary GeoDataFrame
minx, miny, maxx, maxy = WB6_boundary_dissolved_reproj.total_bounds

# Create bounding_box_dict with correct keys for downstream use
bounding_box_dict = {
    "minx": float(minx),
    "miny": float(miny),
    "maxx": float(maxx),
    "maxy": float(maxy)
}

- Create cells' instance for plotting (CRS-m)

In [ ]:
if WB6_cells.crs != CRS_m:
    WB6_cells_plot = WB6_cells.to_crs(CRS_m)
else:
    WB6_cells_plot = WB6_cells

- Plot combined Availability

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib as mpl

# Convert to percent
WB6_cells_plot["LandAvailability_solar_pct"] = WB6_cells_plot["LandAvailability_solar"] * 100
WB6_cells_plot["LandAvailability_wind_pct"] = WB6_cells_plot["LandAvailability_wind"] * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 6), dpi=300)
fig.suptitle(f"{country_name}: Land Availability for VRE development",
             fontsize=16, fontweight="bold", y=0.99)

plot_specs = [
    ("LandAvailability_solar_pct", "Solar", axes[0]),
    ("LandAvailability_wind_pct", "Wind", axes[1]),
]

cmap = plt.cm.get_cmap("YlGn",6)
norm = mpl.colors.Normalize(vmin=0, vmax=100)

for col, panel_title, ax in plot_specs:
    WB6_cells_plot.plot(
        column=col,
        cmap=cmap,
        edgecolor="white",
        linewidth=0.3,
        legend=False,
        vmin=0,
        vmax=100,
        ax=ax,
    )

    WB6_boundary_dissolved_reproj.plot(
        color="none",
        edgecolor="k",
        linewidth=0.5,
        ax=ax
    )

    for _, row in WB6_boundary_dissolved_reproj.iterrows():
        point = row.geometry.representative_point()
        txt = ax.annotate(
            row["Country"],
            xy=(point.x, point.y),
            ha="center",
            va="center",
            fontsize=11,
            fontweight="bold",
            color="black"
        )
        txt.set_path_effects([
            pe.withStroke(linewidth=2.5, foreground="white")
        ])

    ax.set_title(panel_title, fontsize=14, fontweight="bold")
    ax.set_axis_off()

# Manually adjust map area to leave room at bottom
fig.subplots_adjust(bottom=0.1,wspace=0.08)

# Dedicated colorbar axis: [left, bottom, width, height]
cax = fig.add_axes([0.2, 0.02, 0.6, 0.025])

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
cbar.set_label("Land availability (%)", fontsize=14, fontweight="bold")
cbar.ax.tick_params(labelsize=12)
cbar.set_ticks([0, 20, 40, 60, 80, 100])

plt.savefig(
    f"{vis_save_to_root}/{country_name}_map_LandAvailability_solar_wind.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# Attribute Maps

## Load Capacity and Scores

In [ ]:
if existing_VREs_gdf is not None and not existing_VREs_gdf.empty:
    if existing_VREs_gdf.crs != CRS_m:
        existing_VREs_plot = existing_VREs_gdf.to_crs(CRS_m)
    else:
        existing_VREs_plot = existing_VREs_gdf

### Aggregated Capacity

- Calculate

In [ ]:
from RES.CellCapacityProcessor import get_sub_nationally_aggregated_capacity

WB6_cells_aggr = get_sub_nationally_aggregated_capacity(WB6_cells, 
                                                        'Country')
# WB6_cells_sum = WB6_cells.groupby('Country').sum(numeric_only=True)
# Map potential_capacity_solar and potential_capacity_wind to each country and get geometry
WB6_capacity_map = WB6_boundary_dissolved.copy()
WB6_capacity_map["potential_capacity_solar_GW"] = WB6_capacity_map["Country"].map(WB6_cells_aggr["potential_capacity_solar"])/1E3
WB6_capacity_map["potential_capacity_wind_GW"] = WB6_capacity_map["Country"].map(WB6_cells_aggr["potential_capacity_wind"])/1E3
WB6_capacity_map[["Country", "potential_capacity_solar_GW", "potential_capacity_wind_GW", "geometry"]]

- plot

In [ ]:
if WB6_capacity_map.crs != CRS_m:
    WB6_capacity_map_plot = WB6_capacity_map.to_crs(CRS_m)
else:
    WB6_capacity_map_plot = WB6_capacity_map

In [ ]:
import matplotlib.patheffects as pe

# ========= Create subplots =========
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), dpi=500)
Country_name_Y_adjustment:float=12E3 #in meters for CRS_m


# ========= Plot solar capacity =========
WB6_capacity_map_plot.plot(
    column="potential_capacity_solar_GW",
    cmap="YlOrRd",
    linewidth=0.8,
    edgecolor="k",
    legend=True,
    ax=ax1,
    legend_kwds={"label": "Solar Potential (GW)", "shrink": 0.7}
)
ax1.set_title("Solar Potential Capacity (GW)", fontsize=15, weight="bold")
ax1.set_axis_off()

# ========= Annotate numbers and country names with white halo =========
for idx, row in WB6_capacity_map_plot.iterrows():
    centroid = row.geometry.centroid
    val = row["potential_capacity_solar_GW"]
    # Capacity value
    ax1.annotate(
        f"{val:,.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=11,
        ha="center",
        va="center",
        fontweight="bold",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )
    # ========= Country name slightly above =========
    ax1.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
        color="black",
        fontsize=10,
        ha="center",
        va="bottom",
        fontweight="normal",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )

# ========= Plot wind capacity ==============
WB6_capacity_map_plot.plot(
    column="potential_capacity_wind_GW",
    cmap="BuPu",
    linewidth=0.8,
    edgecolor="k",
    legend=True,
    ax=ax2,
    legend_kwds={"label": "Wind Potential (GW)", "shrink": 0.7}
)
ax2.set_title("Wind Potential Capacity (GW)", fontsize=15, weight="bold")
ax2.set_axis_off()

# ========= Annotate numbers and country names with white halo =========
for idx, row in WB6_capacity_map_plot.iterrows():
    centroid = row.geometry.centroid
    val = row["potential_capacity_wind_GW"]
    # Capacity value
    ax2.annotate(
        f"{val:,.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=11,
        ha="center",
        va="center",
        fontweight="bold",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )
    # ========= Country name slightly above =========
    ax2.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
        color="black",
        fontsize=10,
        ha="center",
        va="bottom",
        fontweight="normal",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )

# Add existing VREs to the plot

existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    existing_VREs_gdf=existing_VREs_gdf_solar,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)
existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf_wind,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)

fig.legend(handles=solar_legends+wind_legends, loc='upper center', bbox_to_anchor=(0.45, 0.88), ncol=2, fontsize=8, frameon=False)

plt.tight_layout()
plt.savefig(vis_save_to_root/f"{country_kwd}_Capacity_by_Country.png", bbox_inches='tight', transparent=False)

### Aggregated Capacity with LCOE thresholds

In [ ]:
WB6_cells_plot_solar=WB6_cells_plot[WB6_cells_plot['lcoe_solar'] <= 60]
WB6_cells_plot_wind=WB6_cells_plot[WB6_cells_plot['lcoe_wind'] <= 80]
solar_capacity_haircut=0.8
wind_capacity_haircut=0.6

In [ ]:
from RES.CellCapacityProcessor import get_sub_nationally_aggregated_capacity

WB6_cells_solar_aggr = get_sub_nationally_aggregated_capacity(WB6_cells_plot_solar, 
                                                        'Country')
WB6_cells_wind_aggr = get_sub_nationally_aggregated_capacity(WB6_cells_plot_wind, 
                                                        'Country')
# WB6_cells_sum = WB6_cells.groupby('Country').sum(numeric_only=True)
# Map potential_capacity_solar and potential_capacity_wind to each country and get geometry
WB6_capacity_with_threshold_map = WB6_boundary_dissolved_reproj.copy()
WB6_capacity_with_threshold_map["potential_capacity_solar_GW"] = WB6_capacity_with_threshold_map["Country"].map(WB6_cells_solar_aggr["potential_capacity_solar"])/1E3 * solar_capacity_haircut
WB6_capacity_with_threshold_map["potential_capacity_wind_GW"] = WB6_capacity_with_threshold_map["Country"].map(WB6_cells_wind_aggr["potential_capacity_wind"])/1E3*wind_capacity_haircut
WB6_capacity_with_threshold_map[["Country", "potential_capacity_solar_GW", "potential_capacity_wind_GW", "geometry"]]

#### Assumption 2 no haircuts

In [ ]:
WB6_cells_plot_solar=WB6_cells_plot[WB6_cells_plot['lcoe_solar'] <= 65]
WB6_cells_plot_wind=WB6_cells_plot[WB6_cells_plot['lcoe_wind'] <= 80]
solar_capacity_haircut=0.8
wind_capacity_haircut=0.6
from RES.CellCapacityProcessor import get_sub_nationally_aggregated_capacity

WB6_cells_solar_aggr = get_sub_nationally_aggregated_capacity(WB6_cells_plot_solar, 
                                                        'Country')
WB6_cells_wind_aggr = get_sub_nationally_aggregated_capacity(WB6_cells_plot_wind, 
                                                        'Country')
# WB6_cells_sum = WB6_cells.groupby('Country').sum(numeric_only=True)
# Map potential_capacity_solar and potential_capacity_wind to each country and get geometry
WB6_capacity_with_threshold_map = WB6_boundary_dissolved_reproj.copy()
WB6_capacity_with_threshold_map["potential_capacity_solar_GW"] = WB6_capacity_with_threshold_map["Country"].map(WB6_cells_solar_aggr["potential_capacity_solar"])/1E3 * solar_capacity_haircut
WB6_capacity_with_threshold_map["potential_capacity_wind_GW"] = WB6_capacity_with_threshold_map["Country"].map(WB6_cells_wind_aggr["potential_capacity_wind"])/1E3*wind_capacity_haircut
WB6_capacity_with_threshold_map[["Country", "potential_capacity_solar_GW", "potential_capacity_wind_GW", "geometry"]]

- plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

# =============================
# 1️⃣ Create Figure and Subplots
# =============================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), dpi=1000)
# fig.suptitle(f"Resources and Capacity by Country – {country_name}", fontsize=18, fontweight='bold')

# =============================
# 2️⃣ Plot resource maps (from first block)
# =============================
vis.get_data_in_map_plot(
    WB6_cells_plot_solar, 
    resource_type='solar',
    datafield='score',
    compass_size=12,
    ax=ax1, 
    score_threshold=250,
    show=False
)

vis.get_data_in_map_plot(
    WB6_cells_plot_wind, 
    resource_type='wind',
    datafield='score',
    ax=ax2, 
    score_threshold=250,
    show=False
)

# =============================
# 3️⃣ Overlay “Cells without suitable land”
# =============================
WB6_cells_plot.plot(ax=ax1, color='gray', alpha=0.7, zorder=1)
WB6_cells_plot.plot(ax=ax2, color='gray', alpha=0.6, zorder=1)

# Create legend patch
no_land_patch = mpatches.Patch(
    facecolor='gray',
    edgecolor='lightgray',
    alpha=0.7,
    label='economically unfeasible or no developable land'
)

# =============================
# 4️⃣ Add existing solar/wind projects
# =============================
existing_VREs_gdf_solar = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'solar']
ax1, solar_legends = vis.get_existing_committed_VRE_plot(
    ax=ax1,
    existing_VREs_gdf=existing_VREs_gdf_solar,
    existing_VRE_type_column='Technology',
    target_crs=CRS_m,
    marker_scale_existing=14,
    marker_highlight_width=3
)

existing_VREs_gdf_wind = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'wind']
ax2, wind_legends = vis.get_existing_committed_VRE_plot(
    ax=ax2,
    existing_VREs_gdf=existing_VREs_gdf_wind,
    existing_VRE_type_column='Technology',
    target_crs=CRS_m,
    marker_scale_existing=14,
    marker_highlight_width=3
)

# =============================
# 5️⃣ Overlay country boundaries and labels (from second block)
# =============================
Country_name_Y_adjustment = 12E3  # meters in CRS_m


# Add text annotations for capacity and country name
for idx, row in WB6_capacity_with_threshold_map.iterrows():
    centroid = row.geometry.centroid

    # Solar capacity on left
    ax1.annotate(
        f"{row['potential_capacity_solar_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )
    ax1.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

    # Wind capacity on right
    ax2.annotate(
        f"{row['potential_capacity_wind_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )
    ax2.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

# =============================
# 6️⃣ Add legends and note
# =============================
all_legends = solar_legends + wind_legends + [no_land_patch]
fig.legend(
    handles=all_legends,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.88),
    ncol=1,
    fontsize=8,
    frameon=False
)
# Plot only boundaries (no fill)
WB6_capacity_with_threshold_map.boundary.plot(ax=ax1, color='black', linewidth=0.9, zorder=6)
WB6_capacity_with_threshold_map.boundary.plot(ax=ax2, color='black', linewidth=0.6, zorder=6)

plt.tight_layout()
plt.savefig(vis_save_to_root/"Resources_and_Capacity_Combined_with_LCOE_Thresholds.png", bbox_inches='tight', transparent=False)


### Capacity Factor

* Individual Maps

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6), dpi=1000)

vis.get_data_in_map_plot(WB6_cells_plot, 
                resource_type='solar',
                datafield='CF',
                ax=ax1, 
                show=False)
vis.get_data_in_map_plot(WB6_cells_plot, 
                resource_type='wind',
                datafield='CF',
                ax=ax2, 
                show=False)
# ========= Country name slightly above =========
for idx, row in WB6_capacity_map_plot.iterrows():
    centroid = row.geometry.centroid
    val = row["potential_capacity_wind_GW"]
    ax1.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
        color="black",
        fontsize=10,
        ha="center",
        va="bottom",
        fontweight="normal",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )
    ax2.annotate(
            row["Country"],
            (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
            color="black",
            fontsize=10,
            ha="center",
            va="bottom",
            fontweight="normal",
            path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
        )
# Add existing VREs to the plot

existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    target_crs=CRS_m,
                                                    existing_VREs_gdf=existing_VREs_gdf_solar,
                                                    existing_VRE_type_column='Technology',
                                                    marker_scale_existing=15,
                                                    marker_highlight_width=2)

existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf_wind,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                     marker_scale_existing=14,
                                                    marker_highlight_width=2)

fig.legend(handles=solar_legends+wind_legends, loc='upper center', bbox_to_anchor=(0.5, 0.8), ncol=1, fontsize=8, frameon=False)

vis.add_compass_arrow_custom(ax1, text_offset=0.04)
vis.add_compass_arrow_custom(ax2, text_offset=0.04)
plt.tight_layout()
plt.savefig(vis_save_to_root/"Resources_combined_CF.png", bbox_inches='tight', transparent=False)

### Capacity

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6), dpi=1000)
# for double coulmn figure at 500 Dpi (72points per inch) minimum wide required is 3750 px, hence figsize=(7.5, 5) is recommended

# fig.suptitle(f"Resources for {country_name}", fontsize=18, fontweight='bold')

vis.get_data_in_map_plot(WB6_cells_plot, 
                resource_type='solar',
                datafield='capacity',
                ax=ax1, 
                show=False)
vis.get_data_in_map_plot(WB6_cells_plot, 
                resource_type='wind',
                datafield='capacity',
                ax=ax2, 
                show=False)
# ========= Country name slightly above =========
for idx, row in WB6_capacity_map_plot.iterrows():
    centroid = row.geometry.centroid
    val = row["potential_capacity_wind_GW"]
    ax1.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
        color="black",
        fontsize=10,
        ha="center",
        va="bottom",
        fontweight="normal",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )
    ax2.annotate(
            row["Country"],
            (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
            color="black",
            fontsize=10,
            ha="center",
            va="bottom",
            fontweight="normal",
            path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
        )
# Add existing VREs to the plot
existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    existing_VREs_gdf=existing_VREs_gdf_solar,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)
existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf_wind,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)

fig.legend(handles=solar_legends+wind_legends, loc='upper center', bbox_to_anchor=(0.5, 0.88), ncol=1, fontsize=8, frameon=False)

# vis.add_compass_arrow_custom(ax1, text_offset=0.04)
vis.add_compass_arrow_custom(ax2, text_offset=0.04)
plt.tight_layout()
plt.savefig(vis_save_to_root/"Resources_combined_CAPACITY.png", bbox_inches='tight', transparent=False)

### Score

In [ ]:
WB6_cells_clean_solar = WB6_cells_plot[(WB6_cells_plot['potential_capacity_solar'] >= 1) &(WB6_cells_plot['solar_CF_mean'] > 0)]
WB6_cells_clean_wind = WB6_cells_plot[(WB6_cells_plot['potential_capacity_wind'] >= 3) &(WB6_cells_plot['wind_CF_mean'] > 0)]

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6), dpi=1000)
# for double coulmn figure at 500 Dpi (72points per inch) minimum wide required is 3750 px, hence figsize=(7.5, 5) is recommended

fig.suptitle(f"Resources for {country_name}", fontsize=18, fontweight='bold')

vis.get_data_in_map_plot(WB6_cells_clean_solar, 
                resource_type='solar',
                datafield='score',
                compass_size=12,
                ax=ax1, 
                score_threshold=250,
                show=False)

vis.get_data_in_map_plot(WB6_cells_clean_wind, 
                resource_type='wind',
                datafield='score',
                ax=ax2, 
                score_threshold=250,
                show=False,)
# ========= Country name slightly above =========
for idx, row in WB6_capacity_map_plot.iterrows():
    centroid = row.geometry.centroid
    val = row["potential_capacity_wind_GW"]
    ax1.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
        color="black",
        fontsize=10,
        ha="center",
        va="bottom",
        fontweight="normal",
        path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
    )
    ax2.annotate(
            row["Country"],
            (centroid.x, centroid.y + Country_name_Y_adjustment), # in meters for CRS_m
            color="black",
            fontsize=10,
            ha="center",
            va="bottom",
            fontweight="normal",
            path_effects=[pe.withStroke(linewidth=3, foreground="white",alpha=0.6)]
        )
fig.text(
    0.5, -0.05,
    "Note: The Scoring is calculated to reflect Dollar investment required to get a unit of Energy yield (MWh).To reflect market competitiveness and incentives, the Score (CAD/MWh) needs financial adjustment factors to be considered on top of it. Score higher than 200 $/MWh are assumed to be not feasible and not shown in this map.",
    ha='center', va='top', fontsize=7, color='gray',wrap=True,
)# Add existing VREs to the plot
existing_VREs_gdf_solar=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='solar']
ax1,solar_legends=vis.get_existing_committed_VRE_plot(ax=ax1,
                                                    existing_VREs_gdf=existing_VREs_gdf_solar,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)
existing_VREs_gdf_wind=existing_VREs_plot[existing_VREs_plot['Technology'].str.lower()=='wind']
ax2,wind_legends=vis.get_existing_committed_VRE_plot(ax=ax2,
                                                    existing_VREs_gdf=existing_VREs_gdf_wind,
                                                    existing_VRE_type_column='Technology',
                                                    target_crs=CRS_m,
                                                    marker_scale_existing=14,
                                                    marker_highlight_width=3)

# Add all cells in light grey to indicate area without suitable land
WB6_cells_plot.plot(ax=ax1, color='gray', alpha=1,zorder=1) 
WB6_cells_plot.plot(ax=ax2, color='gray', alpha=1,zorder=1)

# Create legend patch for cells without suitable land
no_land_patch = mpatches.Patch(
    facecolor='gray',
    edgecolor='lightgray',
    alpha=0.5,
    label='Cells without suitable land'
)

# Combine with existing handles
all_legends = solar_legends + wind_legends + [no_land_patch]

fig.legend(handles=all_legends, loc='upper center', bbox_to_anchor=(0.5, 0.88), ncol=1, fontsize=8, frameon=False)

plt.tight_layout()

plt.savefig(vis_save_to_root/"Resources_combined_SCORE.png", bbox_inches='tight', transparent=False)

- Score with Capacity

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

# =============================
# 1️⃣ Create Figure and Subplots
# =============================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), dpi=1000)
# fig.suptitle(f"Resources and Capacity by Country – {country_name}", fontsize=18, fontweight='bold')

# =============================
# 2️⃣ Plot resource maps (from first block)
# =============================
vis.get_data_in_map_plot(
    WB6_cells_clean_solar, 
    resource_type='solar',
    datafield='score',
    compass_size=12,
    ax=ax1, 
    score_threshold=250,
    show=False
)

vis.get_data_in_map_plot(
    WB6_cells_clean_wind, 
    resource_type='wind',
    datafield='score',
    ax=ax2, 
    score_threshold=250,
    show=False
)

# =============================
# 3️⃣ Overlay “Cells without suitable land”
# =============================
WB6_cells_plot.plot(ax=ax1, color='gray', alpha=1, zorder=1)
WB6_cells_plot.plot(ax=ax2, color='gray', alpha=1, zorder=1)

# Create legend patch
no_land_patch = mpatches.Patch(
    facecolor='gray',
    edgecolor='lightgray',
    alpha=0.5,
    label='Cells without suitable land'
)

# =============================
# 4️⃣ Add existing solar/wind projects
# =============================
existing_VREs_gdf_solar = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'solar']
ax1, solar_legends = vis.get_existing_committed_VRE_plot(
    ax=ax1,
    existing_VREs_gdf=existing_VREs_gdf_solar,
    existing_VRE_type_column='Technology',
    target_crs=CRS_m,
    marker_scale_existing=14,
    marker_highlight_width=3
)

existing_VREs_gdf_wind = existing_VREs_plot[existing_VREs_plot['Technology'].str.lower() == 'wind']
ax2, wind_legends = vis.get_existing_committed_VRE_plot(
    ax=ax2,
    existing_VREs_gdf=existing_VREs_gdf_wind,
    existing_VRE_type_column='Technology',
    target_crs=CRS_m,
    marker_scale_existing=14,
    marker_highlight_width=3
)

# =============================
# 5️⃣ Overlay country boundaries and labels (from second block)
# =============================
Country_name_Y_adjustment = 12E3  # meters in CRS_m

# Plot only boundaries (no fill)
WB6_capacity_map_plot.boundary.plot(ax=ax1, color='black', linewidth=0.6, zorder=3)
WB6_capacity_map_plot.boundary.plot(ax=ax2, color='black', linewidth=0.6, zorder=3)

# Add text annotations for capacity and country name
for idx, row in WB6_capacity_map_plot.iterrows():
    centroid = row.geometry.centroid

    # Solar capacity on left
    ax1.annotate(
        f"{row['potential_capacity_solar_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )
    ax1.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

    # Wind capacity on right
    ax2.annotate(
        f"{row['potential_capacity_wind_GW']:.1f}",
        (centroid.x, centroid.y),
        color="black",
        fontsize=10,
        ha="center",
        va="center",
        fontweight="bold",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )
    ax2.annotate(
        row["Country"],
        (centroid.x, centroid.y + Country_name_Y_adjustment),
        color="black",
        fontsize=9,
        ha="center",
        va="bottom",
        zorder=4,
        path_effects=[pe.withStroke(linewidth=3, foreground="white", alpha=0.6)]
    )

# =============================
# 6️⃣ Add legends and note
# =============================
all_legends = solar_legends + wind_legends + [no_land_patch]
fig.legend(
    handles=all_legends,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.88),
    ncol=1,
    fontsize=8,
    frameon=False
)

fig.text(
    0.5, -0.05,
    "Note: The Scoring reflects relative investment per MWh yield. Values above 250 $/MWh are considered non-feasible. "
    "Country-level potentials (in GW) are annotated from aggregated site capacities.",
    ha='center',
    va='top',
    fontsize=9,
    color='gray',
    wrap=True,
)

plt.tight_layout()
plt.savefig(vis_save_to_root/"Resources_and_Capacity_Combined.png", bbox_inches='tight', transparent=False)


# Interactive Map

- Cleanup Solar and Wind cells

In [ ]:
m = vis.make_lcoe_map(
    wind_gdf=WB6_cells,
    solar_gdf=WB6_cells,
    save_path=vis_save_to_root / f"WB6_lcoe_map_{RUN_ID}.html",
    basemap_tiles="Esri WorldGrayCanvas",
    wind_lcoe_max=100,
    solar_lcoe_max=65,
)